In [1]:
import os
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_DATASETS_OFFLINE'] = '1'

In [2]:
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset

2026-02-24 13:59:29.602852: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/Users/farhanabdurrahmanmusa/Documents/99 Sidehustle/Pak Jojo/IntentMining/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
filepath = "review_dengan_intent.csv"
df = pd.read_csv(filepath)
df.head(5)

,userName,score,content,at,intent
0,Nabila Livia,5,sukaa,2026-02-18 10:36:04,Praise & Gratitude
1,Antok Sumawan,5,ya bagus,2026-02-18 10:32:49,Praise & Gratitude
2,Khayyira Ira,5,aku suka banget sama tiktok ini,2026-02-18 10:32:45,Praise & Gratitude
3,Aksay Subang,1,akun gua entah kenapa di Ben padalah gua kaga ...,2026-02-18 10:31:33,Account Issue
4,kenzo zildane alvaro,1,tolong diperbaiki,2026-02-18 10:30:44,General Request/Complaint


In [4]:
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.intent.value_counts()

Before:  1000
After:  1000


intent
Unlabeled/Noise              277
Praise & Gratitude           271
Account Issue                129
Performance Issue            100
Feature Complaint/Request     90
Technical Bug/Crash           72
General Request/Complaint     38
Monetization/Earning          23
Name: count, dtype: int64

In [5]:
dataset = df.content.values.tolist()

In [6]:
dataset[0:5]

['sukaa',
 'ya bagus',
 'aku suka banget sama tiktok ini',
 'akun gua entah kenapa di Ben padalah gua kaga ngapa ngapain',
 'tolong diperbaiki']

In [7]:
from IndoSBERTEmbedding import IndoSBERTEmbedding

embedding = IndoSBERTEmbedding()
vectors = embedding.getEmbeddings(dataset)

Loading Huggingface model....
Model Loaded.


100%|██████████| 1000/1000 [03:45<00:00,  4.44it/s]


In [8]:
vectors[0]

array([-5.25603473e-01,  3.32280606e-01,  1.15906820e-04,  7.16780126e-03,
       -3.48546714e-01,  5.66146553e-01, -1.64278552e-01,  1.36096060e-01,
        4.64701325e-01, -1.25063986e-01,  1.43200874e-01, -8.04134965e-01,
       -3.73169892e-02,  1.28441285e-02,  6.21265948e-01, -2.59610228e-02,
        4.62310314e-01, -5.78375123e-02, -3.25420499e-01, -4.71290946e-01,
        7.73476779e-01,  2.06139669e-01, -9.32600796e-02,  3.67514193e-02,
       -7.65659437e-02,  1.33830190e-01,  3.22796732e-01, -4.05390114e-01,
        8.88464227e-02,  6.84044242e-01, -6.21281303e-02, -3.38686168e-01,
        1.48874717e-02, -4.33994323e-01,  3.35726529e-01, -4.00579631e-01,
        5.20795763e-01, -3.83500159e-01,  3.51658240e-02, -7.15711296e-01,
        1.68145254e-01,  7.65009046e-01,  5.60266256e-01,  1.62173226e-01,
       -1.10994987e-01, -3.40746850e-01, -4.10492152e-01, -3.72823536e-01,
        3.89402181e-01, -2.32415468e-01,  2.70803869e-01, -7.34851182e-01,
       -1.72696874e-01, -

In [30]:
%%time
model = ITER_DBSCAN(
    initial_distance=0.3,
    initial_minimum_samples=15,
    delta_distance=0.01, 
    delta_minimum_samples=1, 
    max_iteration=15,
    algorithm="IndoBERT", 
    # metric="euclidean"
)

CPU times: user 23 μs, sys: 40 μs, total: 63 μs
Wall time: 304 μs


In [31]:
%%time
labels = model.fit_predict(vectors)

CPU times: user 339 ms, sys: 128 ms, total: 467 ms
Wall time: 145 ms


In [32]:
df['cluster_ids'] = labels
df.cluster_ids.value_counts()

cluster_ids
 2     261
 0     208
-1     172
 1      30
 4      21
 10     15
 21     15
 18     14
 6      13
 3      12
 5      11
 19     10
 22     10
 20      9
 12      9
 7       9
 8       8
 16      8
 9       8
 11      7
 32      6
 13      6
 14      6
 24      6
 15      6
 17      6
 27      6
 45      6
 26      5
 40      5
 28      5
 29      5
 25      5
 23      5
 30      5
 35      5
 47      5
 34      4
 41      4
 33      4
 37      4
 43      4
 31      4
 38      4
 48      4
 36      4
 39      3
 42      3
 44      3
 51      3
 49      3
 46      3
 50      3
Name: count, dtype: int64

In [20]:
df.to_excel("result.xlsx", index=False)

In [13]:
evaluate_dataset = EvaluateDataset(filename=filepath, 
                                   filetype='csv', 
                                   text_column='content', 
                                   target_column='intent')

In [ ]:
parameters = [
{
    "distance": 0.5,
    "minimum_samples":15, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoSBERT",
    "metric": "euclidean"
},
{
    "distance": 0.1,
    "minimum_samples":15, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoSBERT",
    # "metric": "euclidean"
},
{
    "distance": 0.5,
    "minimum_samples":15, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoBERT",
    "metric": "euclidean"
},
{
    "distance": 0.1,
    "minimum_samples":15, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoBERT",
    # "metric": "euclidean"
},
]

In [15]:
%%time
results = evaluate_dataset.evaulate_iter_dbscan(parameters)
result_df = pd.DataFrame.from_dict(results)

  0%|          | 0/4 [00:00<?, ?it/s]

Loading Huggingface model....
Model Loaded.


 25%|██▌       | 1/4 [05:19<15:57, 319.30s/it]

Loading Huggingface model....
Model Loaded.


 50%|█████     | 2/4 [08:56<08:38, 259.42s/it]

Loading HuggingFace model...
Model Loaded.


 75%|███████▌  | 3/4 [10:01<02:50, 170.35s/it]

Loading HuggingFace model...
Model Loaded.


100%|██████████| 4/4 [10:46<00:00, 161.68s/it]

CPU times: user 28min 19s, sys: 3min, total: 31min 19s
Wall time: 10min 46s


In [16]:
result_df

,distance,minimum_samples,delta_distance,delta_minimum_samples,max_iteration,algorithm,metric,time,percentage_labelled,clusters,...,homogeneity_score,completeness_score,normalized_mutual_info_score,adjusted_mutual_info_score,adjusted_rand_score,accuracy,precision,recall,f1,intents
0,0.5,15,0.01,1,15,IndoSBERT,euclidean,0.62,13.6,14,...,0.09,0.16,0.11,0.11,0.10,0.392,28.4,39.2,30.4,3
1,0.1,15,0.01,1,15,IndoSBERT,NaN,0.52,31.2,41,...,0.25,0.31,0.28,0.27,0.30,0.524,49.0,52.4,48.9,6
2,0.5,15,0.01,1,15,IndoBERT,euclidean,0.92,13.6,14,...,0.08,0.17,0.10,0.10,0.05,0.371,31.5,37.1,28.3,3
3,0.1,15,0.01,1,15,IndoBERT,NaN,0.78,41.4,45,...,0.24,0.27,0.26,0.25,0.27,0.499,56.7,49.9,49.1,7
